In [55]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
import warnings
import nltk
from nltk.corpus import stopwords
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow import keras
import pickle

warnings.filterwarnings('ignore')

In [ ]:
print('Libraries imported successfully.')

Libraries imported successfully.


**01.load data**

In [ ]:
emotion_df = pd.read_csv('data/data/emotions.csv')
emotion_df.head()

,Unnamed: 0,text,label
0,0,i just feel really helpless and heavy hearted,4
1,1,ive enjoyed being able to slouch about relax a...,0
2,2,i gave up my internship with the dmrg and am f...,4
3,3,i dont know i feel so lost,0
4,4,i am a kindergarten teacher and i am thoroughl...,4


In [ ]:
violence_df = pd.read_csv('data/data/violence.csv')

In [ ]:
hate_df = pd.read_csv('data/data/hate-speach.csv')

**02.data preprocess**

remove unwanted columns

In [ ]:
emotion_df.drop(columns=['Unnamed: 0'], inplace=True)
violence_df.drop(columns=['Tweet_ID'], inplace=True)
hate_df = hate_df[['tweet', 'class']]

In [ ]:
emotion_df.columns, hate_df.columns, violence_df.columns

(Index(['text', 'label'], dtype='object'),
 Index(['tweet', 'class'], dtype='object'),
 Index(['tweet', 'type'], dtype='object'))

rename column names

In [ ]:
hate_df.rename(columns={'tweet': 'text', 'class': 'label'}, inplace=True)
violence_df.rename(columns={'tweet': 'text', 'type': 'label'}, inplace=True)

In [ ]:
emotion_df.columns, hate_df.columns, violence_df.columns

(Index(['text', 'label'], dtype='object'),
 Index(['text', 'label'], dtype='object'),
 Index(['text', 'label'], dtype='object'))

In [ ]:
emotion_df.isna().sum(), violence_df.isna().sum(), hate_df.isna().sum()

(text     0
 label    0
 dtype: int64,
 text     0
 label    0
 dtype: int64,
 text     0
 label    0
 dtype: int64)

get 12000 recodes for each from emotional, violance and hate data

In [ ]:
emotion_df.shape, hate_df.shape, violence_df.shape

((416809, 2), (24783, 2), (39649, 2))

In [ ]:
#emotion dataset
emotion_df['label'].value_counts()

label
1    141067
0    121187
3     57317
4     47712
2     34554
5     14972
Name: count, dtype: int64

In [ ]:
e_df = pd.DataFrame()
for i in range(6):
  subset = emotion_df[emotion_df['label'] == i].sample(n = 2000, random_state=42)
  e_df = pd.concat([e_df, subset])

emotion_df = e_df.copy()


In [ ]:
#violence dataset
violence_df['label'].value_counts()

label
sexual_violence                 32648
Physical_violence                5945
emotional_violence                651
economic_violence                 217
Harmful_Traditional_practice      188
Name: count, dtype: int64

In [ ]:
sexual_violance = violence_df[violence_df['label'] == 'sexual_violence'].sample(n = 4999, random_state=42)
violence_df = violence_df[violence_df['label'] != 'sexual_violence']

In [ ]:
violence_df.shape

(7001, 2)

In [ ]:
violence_df = pd.concat([violence_df, sexual_violance], axis=0)

In [ ]:
violence_df.shape

(12000, 2)

In [ ]:
violence_df['label'].value_counts()

label
Physical_violence               5945
sexual_violence                 4999
emotional_violence               651
economic_violence                217
Harmful_Traditional_practice     188
Name: count, dtype: int64

In [ ]:
#hate speech dataset
hate_df['label'].value_counts()

label
1    19190
2     4163
0     1430
Name: count, dtype: int64

In [ ]:
offensive_speech = hate_df[hate_df['label'] == 1].sample(n=6407, random_state=42)
hate_df = hate_df[hate_df['label'] != 1]

In [ ]:
hate_df.shape

(5593, 2)

In [ ]:
hate_df = pd.concat([hate_df, offensive_speech], axis = 0)

In [ ]:
hate_df.shape

(12000, 2)

remove index

In [ ]:
emotion_df.reset_index(drop=True, inplace=True)
hate_df.reset_index(drop=True, inplace=True)
violence_df.reset_index(drop=True, inplace=True)

encode violence data labels

In [ ]:
label_encoder = LabelEncoder()
violence_df['label'] = label_encoder.fit_transform(violence_df['label'])

In [ ]:
violence_df['label'].unique()

array([1, 3, 0, 2, 4])

remove stopwords

In [ ]:
nltk.download('stopwords')
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


True

In [ ]:
stop_words = set(stopwords.words('english'))

In [ ]:
len(stop_words)

198

In [ ]:
def remove_stopwords(text):
  all_words = nltk.word_tokenize(text)
  filtered_words = [word for word in all_words if word.lower() not in stop_words]
  return ' '.join(filtered_words)

In [ ]:
emotion_df['text'] = emotion_df['text'].apply(remove_stopwords)
violence_df['text'] = violence_df['text'].apply(remove_stopwords)
hate_df['text'] = hate_df['text'].apply(remove_stopwords)

In [ ]:
emotion_df.head()

,text,label
0,ive learned surround women lift leave feeling ...,0
1,already feel crappy upset situation doesnt help,0
2,feel like lost mourned moved past tears relati...,0
3,could write whole lot im feeling crappy dont t...,0
4,always seem feel inadequate,0


tokenize data and create tokenize sequences

In [ ]:
tokernizer = Tokenizer()
tokernizer.fit_on_texts(pd.concat([emotion_df['text'], violence_df['text'], hate_df['text']]))

In [ ]:
emotion_sequence = tokernizer.texts_to_sequences(emotion_df['text'])
violence_sequence = tokernizer.texts_to_sequences(violence_df['text'])
hate_sequence = tokernizer.texts_to_sequences(hate_df['text'])

In [ ]:
emotion_df['text'].iloc[2]

'feel like lost mourned moved past tears relationship'

In [ ]:
emotion_sequence[2:3]

[[1, 5, 321, 11845, 1208, 421, 1094, 386]]

set fixed length to sequences

In [ ]:
max_length = 50
emotion_padded = pad_sequences(emotion_sequence, maxlen=max_length, padding='post')
violence_padded = pad_sequences(violence_sequence, maxlen=max_length, padding='post')
hate_padded = pad_sequences(hate_sequence, maxlen=max_length, padding='post')

In [ ]:
emotion_label = np.array(emotion_df['label'])
violence_label = np.array(violence_df['label'])
hate_label = np.array(hate_df['label'])

**03.Model Definition**

In [ ]:
#prapare deperate input for each dataset
emotion_input = emotion_padded
violence_input = violence_padded
hate_input = hate_padded

input layers

In [ ]:
#defining mulktiple input layers for each task
emotion_input_layer = keras.layers.Input(shape=(max_length,), name='emotion_input')
violence_input_layer = keras.layers.Input(shape=(max_length,), name='violence_input')
hate_input_layer = keras.layers.Input(shape=(max_length,), name='hate_input')

embedding layers

In [ ]:
#shared embedding layer
embedding_layer = keras.layers.Embedding(input_dim= len(tokernizer.word_index) + 1, output_dim=128)

In [ ]:
#apply the embedding layer to each input
emotion_embedding = embedding_layer(emotion_input_layer)
violence_embedding = embedding_layer(violence_input_layer)
hate_embedding = embedding_layer(hate_input_layer)

lstm layers

In [ ]:
#shared LSTM layer
lstm_layer = keras.layers.LSTM(64, return_sequences=True)

In [ ]:
#apply the lstm layer to each embedding layer
emotion_lstm = lstm_layer(emotion_embedding)
violance_lstm = lstm_layer(violence_embedding)
hate_lstm = lstm_layer(hate_embedding)

pooling and dropout layers

In [ ]:
#shared global average pooling layer and dropout layer
shared_pooling = keras.layers.GlobalAveragePooling1D()
shared_dropout = keras.layers.Dropout(0.5)

In [ ]:
#apply the pooling and dropout layer to each embedding layer
emotion_feature = shared_dropout(shared_pooling(emotion_lstm))
violence_feature = shared_dropout(shared_pooling(violance_lstm))
hate_feature = shared_dropout(shared_pooling(hate_lstm))

output layes

In [ ]:
emotion_output = keras.layers.Dense(6, activation='softmax', name='emotion_output')(emotion_feature)
violence_output = keras.layers.Dense(5, activation='softmax', name='violence_output')(violence_feature)
hate_output = keras.layers.Dense(3, activation='softmax', name='hate_output')(hate_feature)

model build and compile

In [ ]:
model = keras.models.Model(
    inputs = [emotion_input_layer, violence_input_layer, hate_input_layer],
    outputs = [emotion_output, violence_output, hate_output]
)

In [ ]:
model.compile(
    optimizer = 'adam',
    loss = {
        'emotion_output' : 'sparse_categorical_crossentropy',
        'violence_output' : 'sparse_categorical_crossentropy',
        'hate_output' : 'sparse_categorical_crossentropy'
    },
    metrics = {
        'emotion_output' : 'accuracy',
        'violence_output' : 'accuracy',
        'hate_output' : 'accuracy'
    }
)

In [ ]:
model.summary()

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 emotion_input (InputLayer)     [(None, 50)]         0           []                               
                                                                                                  
 violence_input (InputLayer)    [(None, 50)]         0           []                               
                                                                                                  
 hate_input (InputLayer)        [(None, 50)]         0           []                               
                                                                                                  
 embedding (Embedding)          (None, 50, 128)      5345152     ['emotion_input[0][0]',          
                                                                  'violence_input[0][0]',     

In [ ]:
model.fit(
    x = {
        'emotion_input' : emotion_input,
        'violence_input' : violence_input,
        'hate_input' : hate_input
    },
    y = {
        'emotion_output' : emotion_label,
        'violence_output' : violence_label,
        'hate_output' : hate_label
    },
    epochs = 10,
    batch_size = 4
)

Epoch 1/10
3000/3000 [==============================] - 73s 22ms/step - loss: 2.1284 - emotion_output_loss: 1.3542 - violence_output_loss: 0.1926 - hate_output_loss: 0.5817 - emotion_output_accuracy: 0.4231 - violence_output_accuracy: 0.9364 - hate_output_accuracy: 0.7937
Epoch 2/10
3000/3000 [==============================] - 62s 21ms/step - loss: 1.0163 - emotion_output_loss: 0.6075 - violence_output_loss: 0.0419 - hate_output_loss: 0.3669 - emotion_output_accuracy: 0.7872 - violence_output_accuracy: 0.9893 - hate_output_accuracy: 0.8738
Epoch 3/10
3000/3000 [==============================] - 58s 19ms/step - loss: 0.5426 - emotion_output_loss: 0.2897 - violence_output_loss: 0.0167 - hate_output_loss: 0.2362 - emotion_output_accuracy: 0.9198 - violence_output_accuracy: 0.9962 - hate_output_accuracy: 0.9167
Epoch 4/10
3000/3000 [==============================] - 57s 19ms/step - loss: 0.3068 - emotion_output_loss: 0.1669 - violence_output_loss: 0.0102 - hate_output_loss: 0.1297 - emotio

In [56]:
# 1. Save the Keras model
model.save('model.h5')

# 2. Save the Tokenizer
with open("tokernizer.pkl", "wb") as f:
    pickle.dump(tokernizer, f)
    
# 3. Save the Label Encoder (for violence categories)
with open("label_encoder.pkl", "wb") as f:
    pickle.dump(label_encoder, f)
    
# 4. Save configuration (like max_length)
config = {'max_length': 50}
with open('config.pkl', 'wb') as handle:
    pickle.dump(config, handle)